## Upload Vector Dataset to Vector Database

### Installing Utilities and Libraries

In [ ]:
%pip install qdrant-client==1.14.3

### Setting up the Environment

In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct
import json

QDRANT_URL = "http://localhost:6333"
COLLECTION_NAME = "carbonops_enterprise_knowledge"

### Upload Vectors to QDrant DB

In [ ]:
qdrant_client = QdrantClient(url = QDRANT_URL)


if not qdrant_client.collection_exists(COLLECTION_NAME):
        qdrant_client.create_collection(
            collection_name=COLLECTION_NAME,
            vectors_config=VectorParams(size=1536, distance=Distance.DOT)
        )
        print(f"Collection '{COLLECTION_NAME}' created.")

        # Load vectors from file
        with open("vector-data.json", "r") as f:
            data = json.load(f)

        points = [
            PointStruct(
                id=i + 1,
                vector=entry["vector"],
                payload={"content": entry["chunk"]}
            ) for i, entry in enumerate(data)
        ]

        qdrant_client.upsert(
            collection_name=COLLECTION_NAME,
            wait=True,
            points=points
        )
        print(f"Upserted {len(points)} vectors.")
else:
        print(f"Collection '{COLLECTION_NAME}' already exists.")